# Reproducing *Evaluation Protocol Matters More Than Model Choice*

Runs the full pipeline for the paper: binary detection with the source-data class prior preserved,
cumulative feature ablation, multi-class attribution, SHAP, and all figures.

Five steps: **clone → install → run → inspect → download**.
No GPU needed — set Runtime → Change runtime type → **CPU**.

## 1. Clone the repository and install pinned dependencies

In [ ]:
!git clone https://github.com/moh1984/ids-metadata-evaluation-protocol.git
%cd ids-metadata-evaluation-protocol

# Versions are pinned: XGBoost in particular is sensitive to build differences,
# and loosening the pins shifts results in the third decimal.
!pip install -q -r requirements.txt

import sklearn, xgboost, shap, numpy, pandas
print("scikit-learn", sklearn.__version__)
print("xgboost     ", xgboost.__version__)
print("shap        ", shap.__version__)

## 2. Check the data

The repository ships the dataset. To use your own, upload it and change `DATA` below;
if your column names differ from Table 2, edit `COLUMN_MAP` at the top of
`src/ids_pipeline.py`.

In [ ]:
import pandas as pd

DATA = "data/cybersecurity_dataset.csv"
SEED = 42

df = pd.read_csv(DATA)
required = ["timestamp", "src_port", "dst_port", "protocol", "bytes_sent",
            "bytes_received", "user_agent", "url", "is_internal_traffic",
            "label", "attack_type"]
missing = [c for c in required if c not in df.columns]

print(f"rows: {len(df):,}   columns: {len(df.columns)}")
print("missing columns:", missing if missing else "none - good to go")
print()
print(df["attack_type"].value_counts())

## 3. Quick check first (~5 min)

`--fast` uses smaller hyperparameter grids. Run it before committing to the full
pipeline: it catches data and column problems in minutes rather than an hour.

In [ ]:
!python src/ids_pipeline.py --data data/cybersecurity_dataset.csv --out results_fast --fast --seed 42

## 4. Full run — every stage

`run_all.sh` executes eight stages in order: the binary pipeline, the feature
ablation, the multi-class and SHAP analysis, the prevalence sensitivity, the
bootstrap intervals and FFNN variants, the imbalance-strategy comparison, the
repeated splits and the figures. Every table and figure in the paper comes out
of this one command.

The seed is passed to each stage that takes one. Two do not:
`prevalence_sensitivity.py` is deterministic, and `repeated_splits.py` uses its
own list of seeds (42, 1, 7, 13, 2024) because varying the split is the point of
that experiment.

Expect **60–120 minutes** on Colab's two CPU cores. Keep the tab open — Colab
disconnects an idle session after about 90 minutes.

In [ ]:
import time

t0 = time.time()
!bash run_all.sh data/cybersecurity_dataset.csv 42
print()
print(f"elapsed: {(time.time() - t0) / 60:.1f} min")

## 5. Inspect the results

In [ ]:
import pandas as pd
import glob
from IPython.display import display, Markdown

for f in sorted(glob.glob("results/tables/*.csv")):
    display(Markdown("### `" + f.split("/")[-1] + "`"))
    display(pd.read_csv(f).round(4))

In [ ]:
from IPython.display import Image, display
import glob

for f in sorted(glob.glob("results/figures/*.png")):
    print(f)
    display(Image(f, width=760))

## 6. Download everything

In [ ]:
from google.colab import files

!zip -qr ids_results.zip results
files.download("ids_results.zip")

---
### Troubleshooting

| Symptom | Fix |
|---|---|
| `missing columns: [...]` | Edit `COLUMN_MAP` at the top of `src/ids_pipeline.py` |
| Session disconnects mid-run | Re-run; or mount Drive and write `results/` there |
| SVM stage very slow | Normal on 2 cores. Shrink the `clf__C` / `clf__gamma` grids in `build_models()` |
| Figures differ in the third decimal | Dependency versions drifted — reinstall from the pinned `requirements.txt` |